In [ ]:
import numpy as np
import pygame
import cv2
import math, random, time

## Algorithm Overview

In [8]:
def render(agents, trailmap, window):
    H, W = len(trailmap), len(trailmap[0])
    trails = (np.array(trailmap) * 255).astype(np.uint8)
    rgb = np.stack([trails]*3, axis=-1)
    
    grid_surf = pygame.Surface((W, H))
    pygame.surfarray.blit_array(grid_surf, rgb.swapaxes(0, 1))
    
    for (x, y, d) in agents:
        grid_surf.set_at((int(x), int(y)), (255, 255, 255))

    pygame.transform.scale(grid_surf, window.get_size(), window)

In [24]:
def generateAgents(n_agents, gridsize):
    agents = []
    for i in range(n_agents):
        x, y = random.random() * gridsize, random.random() * gridsize
        d = random.random() * 2*math.pi
        agents.append([x, y, d])
    return agents

def initializeTrailmap(gridsize):
    return [[0 for x in range(gridsize)] for y in range(gridsize)]

In [20]:
def moveAgents(agents, gridsize):
    for agent in agents:
        x, y, d = agent
        dx, dy = math.cos(d), math.sin(d)
        nx, ny = x + dx, y + dy
        
        if nx < 0 or nx > gridsize-1:
            nx = min(max(nx, 0), gridsize-1)
            d = random.random() * 2*math.pi
        if ny < 0 or ny > gridsize-1:
            ny = min(max(ny, 0), gridsize-1)
            d = random.random() * 2*math.pi
            
        agent[:] = nx, ny, d

In [39]:
def run(n_agents, gridsize, diffusion_speed, evaporation_speed, turn_speed, sensor_distance, sensor_angle):
    agents = generateAgents(n_agents, gridsize)
    trailmap = initializeTrailmap(gridsize)
    
    pygame.init()
    window = pygame.display.set_mode((800, 800))
    clock = pygame.time.Clock()
    try:
        running = True
        while running:
            for e in pygame.event.get():
                if e.type == pygame.QUIT:
                    running = False

            try:
                moveAgents(agents, gridsize)
                leaveTrail(agents, trailmap)
                updateTrail(trailmap, diffusion_speed, evaporation_speed)
                turnAgents(agents, trailmap, turn_speed, sensor_distance, sensor_angle)
            except:
                pass
                    
            render(agents, trailmap, window)
            pygame.display.flip()
            clock.tick(20)
    finally:
        pygame.quit()

In [38]:
config = {
    "n_agents": 20,
    "gridsize": 50,
    "diffusion_speed": 0.3,
    "evaporation_speed": 2/255,
    "turn_speed": 0.5,
    "sensor_distance": 2,
    "sensor_angle": math.pi/4,
}

In [ ]:
# agent movement
run(**config)

In [28]:
def leaveTrail(agents, trailmap):
    for (x, y, d) in agents:
        trailmap[int(y)][int(x)] = 1

In [ ]:
# leaving trail
run(**config)

In [49]:
def updateTrail(trailmap, diffusion_speed, evaporation_speed):
    H, W = len(trailmap), len(trailmap[0])
    old_trailmap = [[trailmap[y][x] for x in range(W)] for y in range(H)]

    for y in range(H):
        for x in range(W):
            # calculate blurred value
            total = 0
            for oy in (-1, 0, 1):
                for ox in (-1, 0, 1):
                    ny, nx = y + oy, x + ox
                    if 0 <= ny < H and 0 <= nx < W:
                        total += old_trailmap[ny][nx]
            blurred_value = total / 9

            # diffusion: blend with neighbors
            value = (1 - diffusion_speed) * old_trailmap[y][x] + diffusion_speed * blurred_value
            # evaporation: reduce to 0
            trailmap[y][x] = max(0, value - evaporation_speed)

In [ ]:
# Diffusion and evaporation
run(**config)

In [50]:
def turnAgents(agents, trailmap, turn_speed, sensor_distance, sensor_angle):
    H, W = len(trailmap), len(trailmap[0])
    
    def sense(agent, trailmap, angle_offset, sensor_distance):
        x, y, d = agent
        angle = d + angle_offset
        sx = int(x + math.cos(angle) * sensor_distance)
        sy = int(y + math.sin(angle) * sensor_distance)
        if 0 <= sy < H and 0 <= sx < W:
            return trailmap[sy][sx]
        return 0
        
    for agent in agents:
        kwargs = {
            "agent": agent,
            "trailmap": trailmap,
            "sensor_distance": sensor_distance,
        }
        forward = sense(**kwargs, angle_offset=0)
        left = sense(**kwargs, angle_offset=-sensor_angle)
        right = sense(**kwargs, angle_offset=sensor_angle)

        r = random.random()
        if forward > left and forward > right: pass # go forward
        elif forward < left and forward < right: agent[2] += (r - 0.5) * turn_speed # turn randomly
        elif left > right: agent[2] -= r * turn_speed # turn left
        elif right > left: agent[2] += r * turn_speed # turn right

In [ ]:
# Turning toward trails
run(**config)

In [59]:
config2 = {
    "n_agents": 300,
    "gridsize": 100,
    "diffusion_speed": 0.3,
    "evaporation_speed": 2/255,
    "turn_speed": 0.5,
    "sensor_distance": 2,
    "sensor_angle": math.pi/4,
}
run(**config2)

In [63]:
config3 = {
    "n_agents": 1500,
    "gridsize": 250,
    "diffusion_speed": 0.3,
    "evaporation_speed": 2/255,
    "turn_speed": 0.5,
    "sensor_distance": 2,
    "sensor_angle": math.pi/4,
}
run(**config3)

## Vectorization

In [ ]:
rng = np.random.default_rng(seed=42)

# agents is an (n_agents, 3) float array, columns: y, x, direction
# trailmap is a (gridsize, gridsize) float array, indexed trailmap[y, x]
# every function changes agents / trailmap in place, so run() works unchanged

def render(agents, trailmap, window):
    H, W = trailmap.shape
    trails = (np.clip(trailmap, 0, 1) * 255).astype(np.uint8)
    rgb = np.stack([trails]*3, axis=-1)

    # draw all agents at once
    rgb[agents[:,0].astype(int), agents[:,1].astype(int)] = 255

    grid_surf = pygame.Surface((W, H))
    pygame.surfarray.blit_array(grid_surf, rgb.swapaxes(0, 1))
    pygame.transform.scale(grid_surf, window.get_size(), window)

def generateAgents(n_agents, gridsize):
    agents = np.zeros((n_agents, 3), dtype=np.float64)

    # spawn in a circle
    center_y, center_x = gridsize / 2, gridsize / 2
    radius = gridsize * 0.2

    theta = rng.random(n_agents) * 2 * math.pi
    r = radius * np.sqrt(rng.random(n_agents))
    agents[:,0] = center_y + r * np.sin(theta)  # y position
    agents[:,1] = center_x + r * np.cos(theta)  # x position
    agents[:,2] = np.arctan2(center_y - agents[:,0], center_x - agents[:,1]) # point to center

    return agents

def initializeTrailmap(gridsize):
    return np.zeros((gridsize, gridsize))

def moveAgents(agents, gridsize):
    agents[:,0] += np.sin(agents[:,2])
    agents[:,1] += np.cos(agents[:,2])

    # agents that left the grid have their direction randomized and position clamped
    out = (agents[:,0] < 0) | (agents[:,0] > gridsize-1) | (agents[:,1] < 0) | (agents[:,1] > gridsize-1)
    agents[out, 2] = rng.random(out.sum()) * 2*math.pi
    agents[:,:2] = np.clip(agents[:,:2], 0, gridsize-1)

def leaveTrail(agents, trailmap):
    trailmap[agents[:,0].astype(int), agents[:,1].astype(int)] = 1

def updateTrail(trailmap, diffusion_speed, evaporation_speed):
    # 3x3 average of neighbors, zeros past the edge
    blurred = cv2.blur(trailmap, (3, 3), borderType=cv2.BORDER_CONSTANT)

    # diffusion: blend with neighbors
    value = (1 - diffusion_speed) * trailmap + diffusion_speed * blurred
    # evaporation: reduce to 0
    trailmap[:] = np.maximum(0, value - evaporation_speed)

def turnAgents(agents, trailmap, turn_speed, sensor_distance, sensor_angle):
    # pad with zeros so sensors past the edge read 0 instead of going out of bounds
    pad = int(sensor_distance) + 1
    padded = np.pad(trailmap, pad)

    # one call senses for all agents
    def sense(angle_offset):
        angle = agents[:,2] + angle_offset
        sy = (agents[:,0] + np.sin(angle) * sensor_distance).astype(int)
        sx = (agents[:,1] + np.cos(angle) * sensor_distance).astype(int)
        return padded[sy + pad, sx + pad]

    forward = sense(0)
    left = sense(-sensor_angle)
    right = sense(sensor_angle)

    random_mask = (forward < left) & (forward < right)
    left_mask   = (left > forward) & (left > right)
    right_mask  = (right > forward) & (right > left)

    agents[random_mask, 2] += (rng.random(random_mask.sum()) - 0.5) * turn_speed
    agents[left_mask, 2]   -= rng.random(left_mask.sum()) * turn_speed
    agents[right_mask, 2]  += rng.random(right_mask.sum()) * turn_speed

In [67]:
config3 = {
    "n_agents": 1500,
    "gridsize": 250,
    "diffusion_speed": 0.3,
    "evaporation_speed": 2/255,
    "turn_speed": 0.5,
    "sensor_distance": 2,
    "sensor_angle": math.pi/4,
}
run(**config3)

In [68]:
config4 = {
    "n_agents": 10000,
    "gridsize": 800,
    "diffusion_speed": 0.3,
    "evaporation_speed": 2/255,
    "turn_speed": 0.5,
    "sensor_distance": 2,
    "sensor_angle": math.pi/4,
}
run(**config4)